# ORVIX: Fine-Tuning an SRE Diagnostic Language Model on LogHub OpenStack Telemetry
This notebook trains a domain-specific Small Language Model (SLM) using **QLoRA (Quantized Low-Rank Adaptation)** on structured OpenStack microservices log data from the **LogHub benchmark**.

### Objective:
Given raw or template-parsed system logs (e.g. `nova-api`, `nova-compute`), the model predicts a structured JSON diagnosis containing **severity, root cause, anomaly flag, and recommended remediation tool**.

In [ ]:
# 1. Install required packages
!pip install -q transformers datasets peft trl accelerate bitsandbytes matplotlib

In [ ]:
# 2. Check GPU Availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Download the OpenStack LogHub Dataset if not uploaded
import os
if not os.path.exists("openstack_logs.csv"):
    !wget -q https://raw.githubusercontent.com/logpai/loghub/master/OpenStack/OpenStack_2k.log_structured.csv -O openstack_logs.csv
    print("Downloaded OpenStack_2k.log_structured.csv from LogHub repository.")
else:
    print("Found local openstack_logs.csv")

In [ ]:
# 4. Data Cleaning, Preprocessing & SRE Ground Truth Annotation
import csv
import json
import re
from datasets import Dataset

def clean_log(content):
    if not content: return ""
    content = re.sub(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", "<UUID>", content, flags=re.IGNORECASE)
    content = re.sub(r"\b[0-9a-f]{32}\b", "<HEX32>", content, flags=re.IGNORECASE)
    content = re.sub(r"\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b", "<IP>", content)
    content = re.sub(r"(/[a-zA-Z0-9_\-\.]+)+", "<PATH>", content)
    content = re.sub(r"req-[0-9a-f\-]+", "<REQ_ID>", content, flags=re.IGNORECASE)
    content = re.sub(r"\b\d+\.\d+\b", "<FLOAT>", content)
    return content.replace('""', '"').strip()

def derive_annotation(level, component, content):
    lvl = (level or "INFO").upper()
    c_low = content.lower()
    if lvl == "WARNING" or "unknown base file" in c_low or "too young to remove" in c_low:
        return "MEDIUM", f"Orphaned or transient base image cache file in {component}", "restart_pod", True
    elif lvl in ("ERROR", "CRITICAL") or "404" in c_low or "failed" in c_low:
        return "HIGH", f"Resource failure in {component}", "restart_service", True
    elif "deleting" in c_low or "destroy" in c_low:
        return "LOW", "Lifecycle VM teardown or cleanup", "verify_recovery", False
    elif "spawning" in c_low or "build" in c_low:
        return "LOW", "Normal VM provisioning and hypervisor resource claim", "verify_recovery", False
    else:
        return "LOW", "Routine API transaction and heartbeat", "none", False

samples = []
with open("openstack_logs.csv", mode="r", encoding="utf-8", errors="replace") as f:
    reader = csv.DictReader(f)
    for row in reader:
        clean = clean_log(row.get("Content", ""))
        comp = row.get("Component", "nova.api")
        lvl = row.get("Level", "INFO")
        sev, cause, tool, is_anom = derive_annotation(lvl, comp, clean)
        
        system_msg = "You are an SRE Diagnostic NLP Model. Return strict JSON with severity, root_cause, recommended_tool, and anomaly."
        user_msg = f"Component: {comp}\nLevel: {lvl}\nLog: {clean}"
        assistant_msg = json.dumps({
            "severity": sev,
            "root_cause": cause,
            "recommended_tool": tool,
            "anomaly": is_anom
        })
        
        prompt = f"<|im_start|>system\n{system_msg}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        samples.append({"text": prompt})

dataset = Dataset.from_list(samples).train_test_split(test_size=0.1, seed=42)
print(f"Train samples: {len(dataset['train'])}, Eval samples: {len(dataset['test'])}")

In [ ]:
# 5. Load Qwen 2.5 1.5B in 4-bit Quantization
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
# 6. Configure LoRA (Low-Rank Adaptation)
from peft import LoraConfig

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# 7. Train the Model with SFTTrainer
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./orvix_sre_finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=25,
    fp16=True,
    save_strategy="no",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=384,
    args=training_args
)

trainer.train()

In [ ]:
# 8. Plot Training & Validation Loss Curves
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss = [x["loss"] for x in log_history if "loss" in x]
eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]

plt.figure(figsize=(9, 4))
plt.plot(train_loss, label="Training Loss (Cross-Entropy)", color="#2563eb", lw=2)
if eval_loss:
    plt.plot(eval_loss, label="Evaluation Loss", color="#dc2626", lw=2)
plt.xlabel("Optimization Step")
plt.ylabel("Loss")
plt.title("Fine-Tuning Convergence on OpenStack Telemetry")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# 9. Test Zero-Shot vs Fine-Tuned Output Inference
test_log = "Component: nova.virt.libvirt.imagecache\nLevel: WARNING\nLog: Unknown base file: <PATH>"
test_prompt = f"<|im_start|>system\nYou are an SRE Diagnostic NLP Model. Return strict JSON with severity, root_cause, recommended_tool, and anomaly.<|im_end|>\n<|im_start|>user\n{test_log}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== PREDICTED STRUCTURED RCA JSON ===")
print(response)